# Regression: Predicting Audi A3 Prices

This notebook trains simple linear regression models and writes expected-price outputs to BigQuery.

It is a baseline notebook: explicit, minimal, and easy for students to extend.

### Functional overview
Inputs: modeling records from BigQuery `vw_regression_dataset`.
Process: filter to configured scope, visualize features, train models, compare metrics, publish expected prices.
Outputs: scatter plots saved to disk and a BigQuery prediction table.
Reason: linear regression provides an interpretable expected-price baseline.

**Objective:** Build linear regression models to estimate expected price.
**Inputs:** Scoped listing data from BigQuery.
**Process:** Apply configured scope, train models, and persist outputs.
**Outputs:** Plots and `fact_expected_price_predictions`.
**Why:** Expected-price outputs are consumed by BI decision support.


## 1. Imports & config

All configuration is grouped at the top to reduce cognitive load.

### Configuration
We import the modeling tools, fix a random seed for reproducibility, and set the BigQuery project/dataset. Output folders are created so saved plots and files have stable locations.

### Model choices
We use simple linear regression for interpretability. Reporting MAE and R2 keeps evaluation simple and aligned with beginner learning goals.

**Objective:** Set imports, random seed, and output paths.
**Inputs:** Modeling libraries and BigQuery identifiers.
**Process:** Import modules and create required directories.
**Outputs:** Configured environment for modeling and reporting.
**Why:** Stable configuration reduces errors and improves reproducibility.


In [ ]:
# Objective: Import libraries, load shared config, and prepare output folders.  # Objective
# Input: Modeling libraries, BigQuery client, and project_config.yaml values.  # Input
# Process: Import modules, parse config, set scope and random seed, create output directory.  # Process
# Output: Initialized environment for regression analysis in a configured scope.  # Output

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

from google.cloud import bigquery


def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / '.git').exists() or (p / 'config' / 'project_config.yaml').exists():
            return p
    return start


def load_project_config(config_path: Path) -> dict:
    config = {}
    if not config_path.exists():
        return config
    for raw_line in config_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or ':' not in line:
            continue
        key, value = line.split(':', 1)
        key = key.strip()
        value = value.strip()
        if value.startswith(('"', "'")) and value.endswith(('"', "'")):
            value = value[1:-1]
        elif value.lower() in ('true', 'false'):
            value = value.lower() == 'true'
        else:
            try:
                value = int(value)
            except ValueError:
                try:
                    value = float(value)
                except ValueError:
                    pass
        config[key] = value
    return config


PROJECT_ROOT = find_repo_root(Path.cwd())
PROJECT_CONFIG = load_project_config(PROJECT_ROOT / 'config' / 'project_config.yaml')

RANDOM_SEED = int(PROJECT_CONFIG.get('random_state', 42))
np.random.seed(RANDOM_SEED)

project_id = str(PROJECT_CONFIG.get('gcp_project_id', 'albertheadofdata101')).strip()
dataset_id = str(PROJECT_CONFIG.get('bq_dataset', 'autoscout_audi_a3_germany')).strip()
make = str(PROJECT_CONFIG.get('make', 'audi')).strip().lower()
model = str(PROJECT_CONFIG.get('model', 'a3')).strip().lower()
country_name = str(PROJECT_CONFIG.get('country', 'germany')).strip().lower()
TAG = f'{make}_{model}_{country_name}'

REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)


## 2. Load modeling data from BigQuery

We pull data directly from BigQuery so students use the same source of truth.

### Data loading
We query `vw_regression_dataset` and apply the configured make/model/country scope from `project_config.yaml`.

### Why explicit scope matters
Using explicit scope keeps the business question controlled and avoids hidden automatic scope changes.

**Objective:** Load the modeling dataset from BigQuery.
**Inputs:** `vw_regression_dataset` and configured scope values.
**Process:** Query the view with scope filters and validate row availability.
**Outputs:** `df` with modeling features and target.
**Why:** Scope control is part of professional analytical delivery.


In [ ]:
# Objective: Load modeling data from BigQuery for the configured scope.  # Objective
# Input: project_id, dataset_id, and configured make/model/country values.  # Input
# Process: Query vw_regression_dataset with explicit scope filters and validate row count.  # Process
# Output: df containing the modeling dataset used for expected-price regression.  # Output

client = bigquery.Client(project=project_id)


def run_query(sql):
    return client.query(sql).to_dataframe()


sql_modeling = f'''
SELECT
  listing_id,
  actual_price_eur AS price_eur,
  mileage_km,
  age_years,
  power_hp,
  make,
  model,
  listing_country
FROM `{project_id}.{dataset_id}.vw_regression_dataset`
WHERE LOWER(make) = '{make}'
  AND LOWER(model) = '{model}'
  AND LOWER(listing_country) = '{country_name}'
  AND actual_price_eur IS NOT NULL AND actual_price_eur > 0
  AND mileage_km IS NOT NULL AND mileage_km > 0
  AND age_years IS NOT NULL AND age_years >= 0
  AND power_hp IS NOT NULL AND power_hp > 0
'''

df = run_query(sql_modeling)

if len(df) == 0:
    sql_modeling_no_country = f'''
    SELECT
      listing_id,
      actual_price_eur AS price_eur,
      mileage_km,
      age_years,
      power_hp,
      make,
      model,
      listing_country
    FROM `{project_id}.{dataset_id}.vw_regression_dataset`
    WHERE LOWER(make) = '{make}'
      AND LOWER(model) = '{model}'
      AND actual_price_eur IS NOT NULL AND actual_price_eur > 0
      AND mileage_km IS NOT NULL AND mileage_km > 0
      AND age_years IS NOT NULL AND age_years >= 0
      AND power_hp IS NOT NULL AND power_hp > 0
    '''
    df = run_query(sql_modeling_no_country)
    print('No rows found for configured country; fallback scope used make/model only.')

if len(df) == 0:
    raise ValueError('No rows available for configured regression scope. Check data and config values.')

print('Rows:', len(df))
print('Scope:', make, model, country_name)


## 3. Quick scatter plots (saved to disk)

Plots are saved for course materials; filenames must remain unchanged.

### Visual checks
Scatter plots show the relationship between price and each feature. These plots are saved for teaching and are not used in the model.

### Why visualize
Scatter plots help verify that relationships are reasonable before modeling.

**Objective:** Visualize relationships between price and features.
**Inputs:** df with price, mileage, age, and power.
**Process:** Plot scatter charts and save them.
**Outputs:** Three PNG plots for documentation.
**Why:** Visual checks help validate feature relationships.


In [ ]:
# Objective: Create and save scatter plots for key features.  # Objective
# Input: df with price, mileage, age, and power columns.  # Input
# Process: Sample rows, plot price vs each feature, save PNG files.  # Process
# Output: Three scatter plot images in the reports directory.  # Output


def plot_feature_scatter(df_plot, x_col, x_label, reports_dir, filename):
    plt.figure(figsize=(8, 5))
    plt.scatter(df_plot[x_col], df_plot['price_eur'], alpha=0.4)
    plt.xlabel(x_label)
    plt.ylabel('Price (EUR)')
    plt.title(f'Price vs {x_label}')
    plt.tight_layout()
    plt.savefig(reports_dir / filename, dpi=150)
    plt.show()


PLOT_MAX_POINTS = 20000
plot_df = df.sample(n=min(PLOT_MAX_POINTS, len(df)), random_state=RANDOM_SEED)

plot_feature_scatter(plot_df, 'mileage_km', 'Mileage (km)', REPORTS_DIR,
                     f'scatter_price_vs_mileage_{TAG}.png')
plot_feature_scatter(plot_df, 'age_years', 'Age (years)', REPORTS_DIR,
                     f'scatter_price_vs_age_{TAG}.png')
plot_feature_scatter(plot_df, 'power_hp', 'Power (hp)', REPORTS_DIR,
                     f'scatter_price_vs_power_{TAG}.png')


## 4. Train-test splits

We keep feature selection explicit and visible to avoid confusion about inputs.

### Train-test split
We split data into train and test sets so we can evaluate model performance on unseen data.

### Why split
A train/test split allows us to estimate how well the model generalizes to unseen data.

**Objective:** Define feature sets and split data for evaluation.
**Inputs:** df with numeric features and price.
**Process:** Create X/y and split into train/test.
**Outputs:** Feature matrices and split datasets.
**Why:** Train/test splits provide unbiased evaluation.


In [ ]:
# Objective: Build feature matrices and a train/test split.  # Objective
# Input: df with numeric features and price target.  # Input
# Process: Create feature subsets and split data for testing.  # Process
# Output: X_* feature sets and train/test split for multivariate model.  # Output

X_mileage = df[['mileage_km']]  # Single-feature dataset for mileage.
X_age = df[['age_years']]  # Single-feature dataset for age.
X_power = df[['power_hp']]  # Single-feature dataset for power.
X_multi = df[['mileage_km', 'age_years', 'power_hp']]  # Multi-feature dataset.
y = df['price_eur']  # Target variable for price.

Xb_train, Xb_test, y_train, y_test = train_test_split(
    X_multi, y, test_size=0.2, random_state=RANDOM_SEED
)  # Split multivariate data into train/test.


## 5. Fit simple regressions (report MAE + R2)

Only MAE and R2 are reported as core regression metrics for this course.

### Model training
We train three single-feature models and one multi-feature model, then report MAE and R2 for each.

### What the models learn
Each univariate model learns a straight-line relationship with price. The multivariate model combines all features.

**Objective:** Train and evaluate linear regression models.
**Inputs:** Train/test splits for each feature set.
**Process:** Fit models and compute MAE and R2.
**Outputs:** Printed metrics for each model.
**Why:** Metrics show predictive performance for comparison.


In [ ]:
# Objective: Train simple regression models and report MAE and R2.  # Objective
# Input: Feature datasets and target y.  # Input
# Process: Fit models, predict on test splits, compute metrics.  # Process
# Output: Printed MAE and R2 values for each model.  # Output

# Mileage-only model  # Model using only mileage.
Xm_train, Xm_test, ym_train, ym_test = train_test_split(
    X_mileage, y, test_size=0.2, random_state=RANDOM_SEED
)  # Split data for mileage model.
model_mileage = LinearRegression()  # Create linear regression model.
model_mileage.fit(Xm_train, ym_train)  # Fit model on training data.

y_mileage_pred = model_mileage.predict(Xm_test)  # Predict on test data.
print('Mileage model:')  # Label output.
print('  R2 test:', r2_score(ym_test, y_mileage_pred))  # Print R2 metric.
print('  MAE test (EUR):', mean_absolute_error(ym_test, y_mileage_pred))  # Print MAE.

# Age-only model  # Model using only age.
Xa_train, Xa_test, ya_train, ya_test = train_test_split(
    X_age, y, test_size=0.2, random_state=RANDOM_SEED
)  # Split data for age model.
model_age = LinearRegression()  # Create linear regression model.
model_age.fit(Xa_train, ya_train)  # Fit model on training data.

y_age_pred = model_age.predict(Xa_test)  # Predict on test data.
print('Age model:')  # Label output.
print('  R2 test:', r2_score(ya_test, y_age_pred))  # Print R2 metric.
print('  MAE test (EUR):', mean_absolute_error(ya_test, y_age_pred))  # Print MAE.

# Power-only model  # Model using only power.
Xp_train, Xp_test, yp_train, yp_test = train_test_split(
    X_power, y, test_size=0.2, random_state=RANDOM_SEED
)  # Split data for power model.
model_power = LinearRegression()  # Create linear regression model.
model_power.fit(Xp_train, yp_train)  # Fit model on training data.

y_power_pred = model_power.predict(Xp_test)  # Predict on test data.
print('Power model:')  # Label output.
print('  R2 test:', r2_score(yp_test, y_power_pred))  # Print R2 metric.
print('  MAE test (EUR):', mean_absolute_error(yp_test, y_power_pred))  # Print MAE.

# Multivariate model  # Model using mileage, age, and power.
model_multi = LinearRegression()  # Create multivariate linear regression.
model_multi.fit(Xb_train, y_train)  # Fit model on training data.

y_multi_pred = model_multi.predict(Xb_test)  # Predict on test data.
print('Multivariate model:')  # Label output.
print('  R2 test:', r2_score(y_test, y_multi_pred))  # Print R2 metric.
print('  MAE test (EUR):', mean_absolute_error(y_test, y_multi_pred))  # Print MAE.


## 6. Compare models on the same test split (lightweight)

The comparison table makes it easy to see which model performs best.

### Comparison table
We build a small DataFrame to compare the models on the same test splits.

### Output table
The comparison table summarizes model performance so students can pick the strongest model.

**Objective:** Summarize model performance in a table.
**Inputs:** Metrics from each model.
**Process:** Assemble metrics into a DataFrame.
**Outputs:** Printed comparison table.
**Why:** A table makes model comparison clearer for students.


In [ ]:
# Objective: Compare model performance in a small summary table.  # Objective
# Input: Predictions and targets from each model.  # Input
# Process: Compute R2 and MAE for each model and assemble a DataFrame.  # Process
# Output: metrics_df printed to the screen.  # Output

metrics = []  # Initialize list of metric dictionaries.

metrics.append({
    'model': 'price ~ mileage',
    'r2_test': r2_score(ym_test, y_mileage_pred),
    'mae_test': mean_absolute_error(ym_test, y_mileage_pred),
})  # Store metrics for mileage model.

metrics.append({
    'model': 'price ~ age',
    'r2_test': r2_score(ya_test, y_age_pred),
    'mae_test': mean_absolute_error(ya_test, y_age_pred),
})  # Store metrics for age model.

metrics.append({
    'model': 'price ~ power',
    'r2_test': r2_score(yp_test, y_power_pred),
    'mae_test': mean_absolute_error(yp_test, y_power_pred),
})  # Store metrics for power model.

metrics.append({
    'model': 'price ~ mileage + age + power',
    'r2_test': r2_score(y_test, y_multi_pred),
    'mae_test': mean_absolute_error(y_test, y_multi_pred),
})  # Store metrics for multivariate model.

metrics_df = pd.DataFrame(metrics)  # Convert metrics list to a DataFrame.
print(metrics_df)  # Display the comparison table.


## 7. Train final model and write predictions to BigQuery

Predictions are persisted to a fixed BigQuery table used downstream.

### Persistence
We fit a final model on all data and save per-listing predictions to BigQuery.

### Final output
The prediction table is saved to BigQuery and can be used for dashboards or later analysis.

**Objective:** Fit a final model and save predictions to BigQuery.
**Inputs:** Full dataset features and target.
**Process:** Train on all rows and write predictions table.
**Outputs:** BigQuery table `fact_expected_price_predictions`.
**Why:** Persisted predictions can be used in downstream analysis.


In [ ]:
# Objective: Train a final model on all data and write expected-price outputs to BigQuery.  # Objective
# Input: Full feature set X_multi and target y.  # Input
# Process: Fit model, generate expected prices, build output table, write to BigQuery.  # Process
# Output: BigQuery table fact_expected_price_predictions replaced with fresh predictions.  # Output

model_full = LinearRegression()
model_full.fit(X_multi, y)

expected_price = model_full.predict(X_multi)

predictions_df = df[['listing_id']].copy()
predictions_df['expected_price_eur'] = expected_price.astype(float)

pred_table = f"{project_id}.{dataset_id}.fact_expected_price_predictions"
job_config = bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE')
client.load_table_from_dataframe(predictions_df, pred_table, job_config=job_config).result()
print('Predictions table replaced:', pred_table)
